In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from config import (CLEAN_DATA_PATH ,RANDOM_STATE ,BINARY_COLS ,MULTI_CAT_COLS ,NUMERIC_COLS,
                    REGRESSION_TARGET ,CLASSIFICATION_TARGET ,DASHBOARD_RESOURCES_PATH ,CLASSIFICATION_MODEL_PATH)

# Preproccsing 
from sklearn.model_selection import train_test_split 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder , PolynomialFeatures

# Models
from sklearn.linear_model import LinearRegression ,Ridge ,Lasso ,LogisticRegression 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC 
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier , RandomForestRegressor ,GradientBoostingRegressor
from xgboost import XGBClassifier 

# Metics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [22]:
df = pd.read_csv(CLEAN_DATA_PATH)

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             7043 non-null   str    
 1   Senior Citizen     7043 non-null   str    
 2   Partner            7043 non-null   str    
 3   Dependents         7043 non-null   str    
 4   Tenure Months      7043 non-null   int64  
 5   Phone Service      7043 non-null   str    
 6   Multiple Lines     7043 non-null   str    
 7   Internet Service   7043 non-null   str    
 8   Online Security    7043 non-null   str    
 9   Online Backup      7043 non-null   str    
 10  Device Protection  7043 non-null   str    
 11  Tech Support       7043 non-null   str    
 12  Streaming TV       7043 non-null   str    
 13  Streaming Movies   7043 non-null   str    
 14  Contract           7043 non-null   str    
 15  Paperless Billing  7043 non-null   str    
 16  Payment Method     7043 non-null   

In [24]:
def make_preprocessor(extra_numeric_cols=None):
    numeric_cols = NUMERIC_COLS + (extra_numeric_cols or [])
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("bin", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), BINARY_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), MULTI_CAT_COLS),
    ])

In [25]:
def make_poly_preprocessor(degree):
    numeric_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(include_bias=False,degree=degree)),
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_pipe, NUMERIC_COLS),
        ("bin", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), BINARY_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), MULTI_CAT_COLS),
    ])

In [26]:
regression_results = []
classification_results = []


In [27]:
def log_regression_result(results_list, model_name, y_true, y_pred):
    row = {
        "Model": model_name,
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }
    results_list.append(row)
    print(f"{model_name:30s} | RMSE: {row['RMSE']:8.2f} | MAE: {row['MAE']:8.2f} | "
          f"R2: {row['R2']:.4f}")
    return row


# Reggression 
### Target value CLTV

Models 
1. Linear
2. Polynomial
2. Lasso
3. Ridge
4. Random forest Reggresion 
5. Gradiant boosting Reggresion

In [28]:
reg_feature_cols = NUMERIC_COLS + BINARY_COLS + MULTI_CAT_COLS
X_reg = df[reg_feature_cols]
y_reg = df[REGRESSION_TARGET]
 
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

In [29]:
def train_poly_regression(X_train, X_test, y_train, y_test, results_list):    
    pipeline = Pipeline([
        ("preprocessor", make_poly_preprocessor(4)),
        ("model", LinearRegression()),
    ])
    pipeline.fit(X_train,y_train)
    y_preds = pipeline.predict(X_test)
    log_regression_result(results_list,'Polynomial Regression',y_test,y_preds)
train_poly_regression(X_reg_train,X_reg_test,y_reg_train,y_reg_test,regression_results)

Polynomial Regression          | RMSE:  1038.58 | MAE:   883.20 | R2: 0.2060


In [30]:
regression_models = {
    "Linear Regression": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", LinearRegression()),
    ]),
    "Lasso Regression": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", Lasso(random_state=RANDOM_STATE, max_iter=10000, alpha=7.0)),
    ]),
    "Ridge Regression": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", Ridge(random_state=RANDOM_STATE , alpha=225.0)),
    ]),
    "Random Forest Regressor": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1,n_estimators=307,min_samples_leaf=1,max_features=1,max_depth=1)),
    ]),
    "Gradient Boosting Regressor": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE,learning_rate=0.008,max_depth=1,n_estimators=335,subsample=1)),
    ]),
}


In [31]:
for name, pipe in regression_models.items():
    pipe.fit(X_reg_train, y_reg_train)
    preds = pipe.predict(X_reg_test)
    log_regression_result(regression_results, name, y_reg_test, preds)


Linear Regression              | RMSE:  1062.44 | MAE:   897.94 | R2: 0.1691
Lasso Regression               | RMSE:  1061.70 | MAE:   898.20 | R2: 0.1702
Ridge Regression               | RMSE:  1062.02 | MAE:   898.43 | R2: 0.1697
Random Forest Regressor        | RMSE:  1145.07 | MAE:   961.42 | R2: 0.0348
Gradient Boosting Regressor    | RMSE:  1026.09 | MAE:   873.30 | R2: 0.2250


# Regression Results

In [32]:
reg_results = pd.DataFrame(regression_results)
reg_results = reg_results.sort_values(by='R2',ascending=False)
reg_results.to_csv(os.path.join(DASHBOARD_RESOURCES_PATH, "regression_results.csv"), index=False)
reg_results

,Model,RMSE,MAE,R2
5,Gradient Boosting Regressor,1026.092964,873.301475,0.224954
0,Polynomial Regression,1038.578161,883.200988,0.205978
2,Lasso Regression,1061.702289,898.197690,0.170226
3,Ridge Regression,1062.019030,898.426632,0.169731
1,Linear Regression,1062.444686,897.939875,0.169065
4,Random Forest Regressor,1145.067029,961.419654,0.034803


# Classification 
### Target value Churn

Models 
1. Logistic
2. KNN
3. SVM
4. Descion tree
5. Random forest
6. Gradiant boosting 

In [33]:
clf_feature_cols = NUMERIC_COLS + BINARY_COLS + MULTI_CAT_COLS +[REGRESSION_TARGET]
X_clf = df[clf_feature_cols]
y_clf = df[CLASSIFICATION_TARGET]
 
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)


In [34]:
def log_classification_result(results_list, model_name, y_true, y_pred, y_proba):
    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        }
    
    results_list.append(row)
    print(f"{model_name:20s} | Acc: {row['Accuracy']:.4f} | Prec: {row['Precision']:.4f} | "
          f"Rec: {row['Recall']:.4f} | F1: {row['F1']:.4f} | AUC: {row['ROC_AUC']:.4f}")
    return row


In [35]:
classification_models = {
    "Logistic Regression":Pipeline([
        ("preprocessor",make_preprocessor([REGRESSION_TARGET])),
        ("model",LogisticRegression(C=12.5,random_state=RANDOM_STATE,class_weight='balanced'))
    ]),
    "KNN": Pipeline([
        ("preprocessor", make_preprocessor([REGRESSION_TARGET])),
        ("model", KNeighborsClassifier(n_neighbors=21,weights="uniform",metric="manhattan")),
    ]),
    "SVM": Pipeline([
        ("preprocessor", make_preprocessor([REGRESSION_TARGET])),
        ("model", SVC(probability=True, random_state=RANDOM_STATE,class_weight="balanced",C=1.0,gamma="scale",kernel="rbf")),
    ]),
    "Naive Bayes": Pipeline([
        ("preprocessor",make_preprocessor([REGRESSION_TARGET])),
        ("model",GaussianNB()),
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", make_preprocessor([REGRESSION_TARGET])),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE,class_weight="balanced",max_depth=6,min_samples_leaf=8)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", make_preprocessor([REGRESSION_TARGET])),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1,class_weight="balanced",
                                         max_depth=10,max_features=0.5,min_samples_leaf=24,n_estimators=472)),
    ]),
    "XG Boosting": Pipeline([
        ("preprocessor", make_preprocessor([REGRESSION_TARGET])),
        ("model", XGBClassifier(random_state=RANDOM_STATE,n_estimators=122,learning_rate=0.008,max_depth=8,subsample=0.63,min_child_weight=5,scale_pos_weight=3),
    )]),
}

In [36]:
for name, pipe in classification_models.items():
    pipe.fit(X_clf_train, y_clf_train)
    preds = pipe.predict(X_clf_test)
    probs = pipe.predict_proba(X_clf_test)[:, 1]
    log_classification_result(classification_results,name,y_clf_test,preds,probs)


Logistic Regression  | Acc: 0.7445 | Prec: 0.5122 | Rec: 0.7834 | F1: 0.6195 | AUC: 0.8479
KNN                  | Acc: 0.7814 | Prec: 0.5868 | Rec: 0.5963 | F1: 0.5915 | AUC: 0.8225
SVM                  | Acc: 0.7580 | Prec: 0.5297 | Rec: 0.7861 | F1: 0.6329 | AUC: 0.8393
Naive Bayes          | Acc: 0.7076 | Prec: 0.4710 | Rec: 0.8262 | F1: 0.6000 | AUC: 0.8167
Decision Tree        | Acc: 0.7516 | Prec: 0.5217 | Rec: 0.7701 | F1: 0.6220 | AUC: 0.8363
Random Forest        | Acc: 0.7736 | Prec: 0.5506 | Rec: 0.7995 | F1: 0.6521 | AUC: 0.8562
XG Boosting          | Acc: 0.7644 | Prec: 0.5376 | Rec: 0.8021 | F1: 0.6438 | AUC: 0.8541


In [ ]:
rf_model = classification_models["Random Forest"]


save_path = Path("Dashboard/Resources/churn_model.pkl")
save_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(rf_model, save_path)

print(f"Random Forest model saved successfully to {save_path}")

# Classification Results

In [37]:
clf_results = pd.DataFrame(classification_results)
clf_results = clf_results.sort_values(by='F1',ascending=False)
clf_results.to_csv(os.path.join(DASHBOARD_RESOURCES_PATH, "classification_results.csv"), index=False)
clf_results

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
5,Random Forest,0.773598,0.550645,0.799465,0.652126,0.856167
6,XG Boosting,0.764372,0.537634,0.802139,0.643777,0.854074
2,SVM,0.757984,0.529730,0.786096,0.632939,0.839335
4,Decision Tree,0.751597,0.521739,0.770053,0.622030,0.836275
0,Logistic Regression,0.744500,0.512238,0.783422,0.619450,0.847924
3,Naive Bayes,0.707594,0.471037,0.826203,0.600000,0.816668
1,KNN,0.781405,0.586842,0.596257,0.591512,0.822504
